In [ ]:
import pandas as pd
import numpy as np

# STEP 1: Load the cleaned data from Week 1
df = pd.read_pickle('../data/online_retail_cleaned.pkl')
print("Successfully loaded Week 1 cleaned data.")

print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
print("\n--- First 3 Rows of Cleaned Data ---")
print(df[['CustomerID', 'InvoiceNo', 'InvoiceDate', 'InvoiceMonth', 'CohortMonth']].head(3))

print("\n--- Data Types of Your Columns ---")
print(df.dtypes)

# STEP 2: The Week 2 Cohort Matrix Logic
def calculate_cohort_matrix(df):
    # Extract integer year and month
    invoice_year, invoice_month = df['InvoiceDate'].dt.year, df['InvoiceDate'].dt.month
    cohort_year, cohort_month = df['CohortMonth'].dt.year, df['CohortMonth'].dt.month

    # Calculate difference in months (Cohort Index)
    years_diff = invoice_year - cohort_year
    months_diff = invoice_month - cohort_month
    
    df['CohortIndex'] = years_diff * 12 + months_diff 

    # Group by CohortMonth and CohortIndex to count unique customers
    grouping = df.groupby(['CohortMonth', 'CohortIndex'])
    cohort_data = grouping['CustomerID'].nunique().reset_index()

    # Pivot table to create the matrix
    cohort_matrix = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
    
    # Calculate retention percentages
    cohort_sizes = cohort_matrix.iloc[:, 0]
    retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0)
    
    return cohort_matrix, retention_matrix

# STEP 3: Run the calculations and save the results for Week 3!
matrix_counts, matrix_pct = calculate_cohort_matrix(df)

matrix_pct.to_pickle('../data/retention_matrix_pct.pkl')
print("Week 2 Matrix generated and saved locally!")

Successfully loaded Week 1 cleaned data.
Total Rows: 397884 | Total Columns: 10

--- First 3 Rows of Cleaned Data ---
   CustomerID InvoiceNo         InvoiceDate InvoiceMonth CohortMonth
0     17850.0    536365 2010-12-01 08:26:00      2010-12     2010-12
1     17850.0    536365 2010-12-01 08:26:00      2010-12     2010-12
2     17850.0    536365 2010-12-01 08:26:00      2010-12     2010-12

--- Data Types of Your Columns ---
InvoiceNo                  str
StockCode                  str
Description                str
Quantity                 int64
InvoiceDate     datetime64[us]
UnitPrice              float64
CustomerID             float64
Country                    str
InvoiceMonth         period[M]
CohortMonth          period[M]
dtype: object
Week 2 Matrix generated and saved locally!
